In [1]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv

In [42]:
class BatsManState(TypedDict):
    runs:int
    balls:int
    fours:int
    sixes:int
    
    sr:float
    bpb:float
    boundary_percentage:float
    summary:str

In [44]:
def calculate_sr(state:BatsManState)->BatsManState:
    sr= state["runs"]/state["balls"]*100
    return {'sr':sr}

In [45]:
def calculate_bpb(state:BatsManState)->BatsManState:
    bpb = state['balls']/(state["fours"]+state["sixes"])
    return {'bpb':bpb}

In [46]:
def calculate_boundary_percentage(state:BatsManState)->BatsManState:
    boundary_percentage = (state["fours"]+state["sixes"])/state["runs"]*100
    return {'boundary_percentage':boundary_percentage}

In [47]:
def summary(state: BatsManState) -> BatsManState:
    summary = f"""
Strike Rate - {state['sr']}
Balls per boundary - {state['bpb']}
Boundary percentage - {state['boundary_percentage']}
"""
    return {'summary':summary}   # ✅ critical
    
    
    
    

In [48]:
graph=StateGraph(BatsManState)

#Nodes
graph.add_node("calculate_sr",calculate_sr)
graph.add_node("calculate_bpb",calculate_bpb)
graph.add_node("calculate_boundary_percentage",calculate_boundary_percentage)
graph.add_node("summary",summary)

#Edges
graph.add_edge(START,"calculate_sr")
graph.add_edge(START,"calculate_bpb")
graph.add_edge(START,"calculate_boundary_percentage")

graph.add_edge("calculate_sr","summary")
graph.add_edge("calculate_bpb","summary")
graph.add_edge("calculate_boundary_percentage","summary")
graph.add_edge("summary",END)


workflows=graph.compile()




In [49]:
intial_state={'runs': 120, 'balls': 80, 'fours': 10, 'sixes': 5}
final_state=workflows.invoke(intial_state)  
print(final_state["summary"])
workflows.invoke(intial_state)


Strike Rate - 150.0
Balls per boundary - 5.333333333333333
Boundary percentage - 12.5



{'runs': 120,
 'balls': 80,
 'fours': 10,
 'sixes': 5,
 'sr': 150.0,
 'bpb': 5.333333333333333,
 'boundary_percentage': 12.5,
 'summary': '\nStrike Rate - 150.0\nBalls per boundary - 5.333333333333333\nBoundary percentage - 12.5\n'}